# Lesson 3B: Neural Networks PyTorch Practical

## Introduction

In Lesson 1B, we achieved 97% accuracy on breast cancer detection using a single neuron (logistic regression). In Lesson 3A, we learned how combining multiple neurons creates powerful pattern detectors. Now we'll implement these ideas in PyTorch to see if we can improve our diagnostic accuracy.

Think of this evolution like upgrading from a single expert doctor to a team of specialists:
- **Logistic Regression**: One expert making decisions based on all features
- **Neural Network**: Multiple specialists, each focusing on different patterns, collaborating for the final diagnosis

In this lesson, we'll:
1. Build our first neural network using PyTorch's powerful abstractions
2. Implement modern techniques: batch normalization, dropout, and learning rate scheduling  
3. Systematically explore different architectures to find the best model
4. Visualize what our networks learn and understand their decision-making
5. Compare performance with our logistic regression baseline

By the end, you'll have practical experience building, training, and evaluating neural networks - skills that transfer directly to more complex deep learning projects.

Let's see if adding depth to our model improves cancer detection!

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Data Preparation Review](#data-preparation-review)
4. [Building Our First Neural Network](#building-our-first-neural-network)
   - [A Simple Two-Layer Network](#a-simple-two-layer-network)
   - [Training Our First Network](#training-our-first-network)
   - [Comparing with Logistic Regression](#comparing-with-logistic-regression)
5. [Modern Neural Network Architecture](#modern-neural-network-architecture)
   - [Adding Batch Normalization](#adding-batch-normalization)
   - [Implementing Dropout](#implementing-dropout)
   - [Flexible Architecture Design](#flexible-architecture-design)
6. [Advanced Training Techniques](#advanced-training-techniques)
   - [Learning Rate Scheduling](#learning-rate-scheduling)
   - [Gradient Clipping](#gradient-clipping)
   - [Weight Decay](#weight-decay)
7. [Systematic Architecture Search](#systematic-architecture-search)
   - [Defining Search Space](#defining-search-space)
   - [Running Experiments](#running-experiments)
   - [Analyzing Results](#analyzing-results)
8. [Interpretability and Visualization](#interpretability-and-visualization)
   - [Hidden Layer Representations](#hidden-layer-representations)
   - [Feature Importance](#feature-importance)
   - [Activation Patterns](#activation-patterns)
9. [Comprehensive Evaluation](#comprehensive-evaluation)
   - [Performance Comparison](#performance-comparison)
   - [Statistical Significance](#statistical-significance)
   - [Error Analysis](#error-analysis)
10. [Production Considerations](#production-considerations)
    - [Model Complexity Trade-offs](#model-complexity-trade-offs)
    - [When to Use Neural Networks](#when-to-use-neural-networks)
11. [Conclusion](#conclusion)
    - [Key Takeaways](#key-takeaways)
    - [Looking Forward](#looking-forward)
    - [Further Reading](#further-reading)

<a name="required-libraries"></a>
## Required Libraries

We'll use the same libraries as Lesson 1B, plus additional tools for visualization and analysis:

| Library | Purpose |
|---------|--------|
| PyTorch | Neural network implementation |
| NumPy | Numerical operations |
| Pandas | Data manipulation |
| Scikit-learn | Data preprocessing and metrics |
| Matplotlib/Seaborn | Visualization |
| Additional | t-SNE for dimensionality reduction |

In [ ]:
# System utilities
from typing import List, Optional, Tuple, Dict, Any
import warnings
import time
from pathlib import Path

# Core data science
import numpy as np
import pandas as pd
from numpy.typing import NDArray

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

# Scikit-learn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score
)
from sklearn.manifold import TSNE

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# Display settings
%matplotlib inline
plt.style.use('seaborn-v0_8')
warnings.filterwarnings('ignore')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLibraries loaded successfully!")

<a name="data-preparation-review"></a>
## Data Preparation Review

We'll use the same data preparation pipeline from Lesson 1B to ensure fair comparison between logistic regression and neural networks:

In [ ]:
# Load and prepare data (same as Lesson 1B)
def load_cancer_data():
    """Load Wisconsin breast cancer dataset."""
    cancer = load_breast_cancer()
    df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
    df['target'] = cancer.target
    return df

def prepare_data(df: pd.DataFrame) -> Tuple[NDArray, NDArray, NDArray, NDArray, NDArray, NDArray, StandardScaler]:
    """Prepare data with train/validation/test split."""
    features = df.drop('target', axis=1).values
    labels = df['target'].values
    
    # First split: separate test set
    train_val_features, test_features, train_val_labels, test_labels = train_test_split(
        features, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Second split: separate train and validation
    train_features, val_features, train_labels, val_labels = train_test_split(
        train_val_features, train_val_labels, test_size=0.2, random_state=42, stratify=train_val_labels
    )
    
    # Standardize features
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)
    test_features_scaled = scaler.transform(test_features)
    
    return (
        train_features_scaled, val_features_scaled, test_features_scaled,
        train_labels, val_labels, test_labels,
        scaler
    )

class CancerDataset(Dataset):
    """PyTorch dataset for cancer data."""
    def __init__(self, features: NDArray, labels: NDArray):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels).reshape(-1, 1)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Load and prepare data
df = load_cancer_data()
print(f"Dataset shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"Target distribution:\n{df['target'].value_counts()}")

# Prepare train/val/test splits
(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    scaler
) = prepare_data(df)

print(f"\nData splits:")
print(f"Training: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

# Create datasets and dataloaders
batch_size = 32

train_dataset = CancerDataset(X_train, y_train)
val_dataset = CancerDataset(X_val, y_val)
test_dataset = CancerDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"\nBatch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")
print(f"Input features: {X_train.shape[1]}")

Let's do a quick reminder of our data characteristics:

In [ ]:
# Quick visualization of feature distributions
def plot_feature_distributions(df, n_features=6):
    """Plot distributions of top features by correlation with target."""
    # Get correlations with target
    correlations = df.corr()['target'].abs().sort_values(ascending=False)[1:n_features+1]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, (feature, corr) in enumerate(correlations.items()):
        ax = axes[idx]
        
        # Plot distributions by class
        for target, color, label in [(0, 'red', 'Malignant'), (1, 'blue', 'Benign')]:
            data = df[df['target'] == target][feature]
            ax.hist(data, bins=30, alpha=0.5, color=color, label=label, density=True)
        
        ax.set_title(f'{feature}\nCorrelation: {corr:.3f}')
        ax.set_xlabel('Value')
        ax.set_ylabel('Density')
        ax.legend()
    
    plt.suptitle('Top 6 Features by Correlation with Target', fontsize=16)
    plt.tight_layout()
    plt.show()

plot_feature_distributions(df)

<a name="building-our-first-neural-network"></a>
## Building Our First Neural Network

<a name="a-simple-two-layer-network"></a>
### A Simple Two-Layer Network

Let's start with a basic neural network that adds just one hidden layer to our logistic regression:

In [ ]:
class SimpleNeuralNetwork(nn.Module):
    """A simple 2-layer neural network for cancer classification.
    
    Architecture:
        Input (30) → Hidden (16) → Output (1)
    
    This adds one hidden layer to logistic regression, allowing
    the model to learn non-linear patterns.
    """
    
    def __init__(self, input_features: int, hidden_size: int = 16):
        super().__init__()
        
        # Layers
        self.fc1 = nn.Linear(input_features, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights using Xavier initialization."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the network.
        
        Args:
            x: Input features [batch_size, input_features]
            
        Returns:
            Predictions [batch_size, 1]
        """
        # Hidden layer
        x = self.fc1(x)
        x = self.relu(x)
        
        # Output layer
        x = self.fc2(x)
        x = self.sigmoid(x)
        
        return x
    
    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Make binary predictions."""
        with torch.no_grad():
            probabilities = self.forward(x)
            return (probabilities > 0.5).float()

# Create and inspect the model
simple_model = SimpleNeuralNetwork(input_features=30, hidden_size=16)
print("Simple Neural Network Architecture:")
print(simple_model)

# Count parameters
total_params = sum(p.numel() for p in simple_model.parameters())
print(f"\nTotal parameters: {total_params}")
print(f"  - fc1: {30 * 16} weights + {16} biases = {30 * 16 + 16}")
print(f"  - fc2: {16 * 1} weights + {1} bias = {16 * 1 + 1}")

# Compare with logistic regression
logistic_params = 30 + 1  # 30 weights + 1 bias
print(f"\nLogistic regression parameters: {logistic_params}")
print(f"Parameter increase: {total_params / logistic_params:.1f}x")

<a name="training-our-first-network"></a>
### Training Our First Network

Let's train this simple network using the same training procedure as Lesson 1B:

In [ ]:
def train_model(model, train_loader, val_loader, epochs=200, lr=0.001, patience=10, device='cpu'):
    """Train a neural network with early stopping.
    
    Args:
        model: PyTorch model to train
        train_loader: Training data loader
        val_loader: Validation data loader
        epochs: Maximum number of epochs
        lr: Learning rate
        patience: Early stopping patience
        device: Device to train on
        
    Returns:
        Tuple of (trained model, training history)
    """
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Training history
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }
    
    # Early stopping
    best_val_loss = float('inf')
    best_model_state = None
    no_improve = 0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_losses = []
        train_correct = 0
        train_total = 0
        
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            
            # Forward pass
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Track metrics
            train_losses.append(loss.item())
            predictions = (outputs > 0.5).float()
            train_correct += (predictions == batch_labels).sum().item()
            train_total += batch_labels.size(0)
        
        # Validation phase
        model.eval()
        val_losses = []
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch_features, batch_labels in val_loader:
                batch_features = batch_features.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_features)
                loss = criterion(outputs, batch_labels)
                
                val_losses.append(loss.item())
                predictions = (outputs > 0.5).float()
                val_correct += (predictions == batch_labels).sum().item()
                val_total += batch_labels.size(0)
        
        # Calculate epoch metrics
        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total
        
        # Store history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        # Print progress
        if (epoch + 1) % 20 == 0:
            print(f'Epoch [{epoch+1}/{epochs}] '
                  f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
    
    # Restore best model
    model.load_state_dict(best_model_state)
    return model, history

# Train the simple neural network
print("Training Simple Neural Network...")
simple_model = SimpleNeuralNetwork(input_features=30, hidden_size=16)
simple_model, simple_history = train_model(
    simple_model, train_loader, val_loader,
    epochs=200, lr=0.001, patience=10, device=device
)

<a name="comparing-with-logistic-regression"></a>
### Comparing with Logistic Regression

Let's train a logistic regression model for comparison:

In [ ]:
# Logistic regression from Lesson 1B
class LogisticRegression(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.linear = nn.Linear(input_features, 1)
        self.sigmoid = nn.Sigmoid()
        
        # Initialize
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)
    
    def forward(self, x):
        return self.sigmoid(self.linear(x))

# Train logistic regression
print("Training Logistic Regression...")
logistic_model = LogisticRegression(input_features=30)
logistic_model, logistic_history = train_model(
    logistic_model, train_loader, val_loader,
    epochs=200, lr=0.001, patience=10, device=device
)

# Compare training curves
def plot_training_comparison(histories, labels):
    """Plot training curves for multiple models."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss curves
    for history, label in zip(histories, labels):
        ax1.plot(history['train_loss'], label=f'{label} (Train)', alpha=0.8)
        ax1.plot(history['val_loss'], label=f'{label} (Val)', linestyle='--', alpha=0.8)
    
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    for history, label in zip(histories, labels):
        ax2.plot(history['train_acc'], label=f'{label} (Train)', alpha=0.8)
        ax2.plot(history['val_acc'], label=f'{label} (Val)', linestyle='--', alpha=0.8)
    
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0.85, 1.0)
    
    plt.tight_layout()
    plt.show()

plot_training_comparison(
    [logistic_history, simple_history],
    ['Logistic Regression', 'Simple NN']
)

Let's evaluate both models on the test set:

In [ ]:
def evaluate_model(model, test_loader, device='cpu'):
    """Evaluate model on test set."""
    model.eval()
    model = model.to(device)
    
    all_predictions = []
    all_probabilities = []
    all_labels = []
    
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            
            probabilities = model(features)
            predictions = (probabilities > 0.5).float()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    # Convert to numpy arrays
    predictions = np.array(all_predictions).flatten()
    probabilities = np.array(all_probabilities).flatten()
    labels = np.array(all_labels).flatten()
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions),
        'roc_auc': roc_auc_score(labels, probabilities)
    }
    
    return metrics, predictions, probabilities, labels

# Evaluate both models
print("Test Set Performance:")
print("=" * 50)

logistic_metrics, _, _, _ = evaluate_model(logistic_model, test_loader, device)
simple_metrics, _, _, _ = evaluate_model(simple_model, test_loader, device)

# Display comparison
comparison_df = pd.DataFrame({
    'Logistic Regression': logistic_metrics,
    'Simple Neural Network': simple_metrics
}).T

print(comparison_df.round(4))
print("\nImprovement over Logistic Regression:")
for metric in comparison_df.columns:
    improvement = (simple_metrics[metric] - logistic_metrics[metric]) * 100
    print(f"{metric}: {improvement:+.2f}%")

<a name="modern-neural-network-architecture"></a>
## Modern Neural Network Architecture

Our simple network showed modest improvements. Let's build a more sophisticated architecture with modern techniques:

<a name="adding-batch-normalization"></a>
### Adding Batch Normalization

Batch normalization helps with:
- Faster training
- Higher learning rates
- Less sensitivity to initialization

<a name="implementing-dropout"></a>
### Implementing Dropout

Dropout prevents overfitting by:
- Randomly dropping neurons during training
- Forcing the network to be robust
- Acting like an ensemble of networks

<a name="flexible-architecture-design"></a>
### Flexible Architecture Design

In [ ]:
class ModernNeuralNetwork(nn.Module):
    """Modern neural network with batch normalization and dropout.
    
    Features:
        - Flexible depth (multiple hidden layers)
        - Batch normalization for stable training
        - Dropout for regularization
        - ReLU activations
    """
    
    def __init__(self, input_features: int, hidden_sizes: List[int] = [64, 32], 
                 dropout_rate: float = 0.3, use_batch_norm: bool = True):
        super().__init__()
        
        self.use_batch_norm = use_batch_norm
        
        # Build layers dynamically
        layers = []
        in_features = input_features
        
        # Hidden layers
        for hidden_size in hidden_sizes:
            # Linear layer
            layers.append(nn.Linear(in_features, hidden_size))
            
            # Batch normalization (if enabled)
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_size))
            
            # Activation
            layers.append(nn.ReLU())
            
            # Dropout
            layers.append(nn.Dropout(dropout_rate))
            
            in_features = hidden_size
        
        # Output layer
        layers.append(nn.Linear(in_features, 1))
        layers.append(nn.Sigmoid())
        
        # Create sequential model
        self.network = nn.Sequential(*layers)
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights with appropriate methods."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                # He initialization for ReLU
                nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the network."""
        return self.network(x)
    
    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Make binary predictions."""
        self.eval()  # Set to evaluation mode (disables dropout)
        with torch.no_grad():
            probabilities = self.forward(x)
            return (probabilities > 0.5).float()

# Create different architectures
architectures = [
    {'name': 'Shallow', 'hidden_sizes': [32], 'dropout': 0.2},
    {'name': 'Medium', 'hidden_sizes': [64, 32], 'dropout': 0.3},
    {'name': 'Deep', 'hidden_sizes': [128, 64, 32], 'dropout': 0.4},
    {'name': 'Wide', 'hidden_sizes': [128, 128], 'dropout': 0.3}
]

# Display architectures
print("Neural Network Architectures:")
print("=" * 50)
for config in architectures:
    model = ModernNeuralNetwork(30, **{k: v for k, v in config.items() if k != 'name'})
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{config['name']} Network:")
    print(f"  Architecture: 30 → {' → '.join(map(str, config['hidden_sizes']))} → 1")
    print(f"  Dropout: {config['dropout']}")
    print(f"  Total parameters: {total_params:,}")

<a name="advanced-training-techniques"></a>
## Advanced Training Techniques

Let's implement modern training techniques that help neural networks train better:

<a name="learning-rate-scheduling"></a>
### Learning Rate Scheduling

Reduces learning rate when validation loss plateaus.

<a name="gradient-clipping"></a>
### Gradient Clipping

Prevents exploding gradients in deep networks.

<a name="weight-decay"></a>
### Weight Decay

L2 regularization to prevent overfitting.

In [ ]:
def train_neural_network(model, train_loader, val_loader, epochs=200, lr=0.001, 
                        weight_decay=1e-4, patience=15, lr_patience=10, 
                        gradient_clip=1.0, device='cpu'):
    """Advanced training function with modern techniques.
    
    Features:
        - Learning rate scheduling
        - Gradient clipping
        - Weight decay (L2 regularization)
        - Early stopping
    """
    model = model.to(device)
    
    # Loss and optimizer
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=lr_patience, verbose=True
    )
    
    # Training history
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'lr': []
    }
    
    # Early stopping
    best_val_loss = float('inf')
    best_model_state = None
    no_improve = 0
    
    # Training loop
    start_time = time.time()
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_losses = []
        train_correct = 0
        train_total = 0
        
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            
            # Forward pass
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            
            # Backward pass with gradient clipping
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            if gradient_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            
            optimizer.step()
            
            # Track metrics
            train_losses.append(loss.item())
            predictions = (outputs > 0.5).float()
            train_correct += (predictions == batch_labels).sum().item()
            train_total += batch_labels.size(0)
        
        # Validation phase
        model.eval()
        val_losses = []
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch_features, batch_labels in val_loader:
                batch_features = batch_features.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_features)
                loss = criterion(outputs, batch_labels)
                
                val_losses.append(loss.item())
                predictions = (outputs > 0.5).float()
                val_correct += (predictions == batch_labels).sum().item()
                val_total += batch_labels.size(0)
        
        # Calculate metrics
        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Print progress
        if (epoch + 1) % 20 == 0:
            elapsed = time.time() - start_time
            print(f'Epoch [{epoch+1}/{epochs}] ({elapsed:.1f}s) '
                  f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f} | '
                  f'LR: {current_lr:.6f}')
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
    
    # Restore best model
    model.load_state_dict(best_model_state)
    
    total_time = time.time() - start_time
    print(f'\nTraining completed in {total_time:.1f} seconds')
    
    return model, history

# Train a modern neural network
print("Training Modern Neural Network...")
modern_model = ModernNeuralNetwork(
    input_features=30,
    hidden_sizes=[64, 32],
    dropout_rate=0.3,
    use_batch_norm=True
)

modern_model, modern_history = train_neural_network(
    modern_model, train_loader, val_loader,
    epochs=300, lr=0.001, weight_decay=1e-4,
    patience=20, lr_patience=10, device=device
)

Let's visualize the advanced training process:

In [ ]:
def plot_advanced_training(history):
    """Plot training curves with learning rate schedule."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    ax = axes[0, 0]
    ax.plot(history['train_loss'], label='Train Loss', alpha=0.8)
    ax.plot(history['val_loss'], label='Val Loss', alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax = axes[0, 1]
    ax.plot(history['train_acc'], label='Train Acc', alpha=0.8)
    ax.plot(history['val_acc'], label='Val Acc', alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Training and Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.85, 1.0)
    
    # Learning rate schedule
    ax = axes[1, 0]
    ax.plot(history['lr'], 'g-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    
    # Gap analysis
    ax = axes[1, 1]
    train_val_gap = np.array(history['train_acc']) - np.array(history['val_acc'])
    ax.plot(train_val_gap, 'r-', linewidth=2)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train-Val Accuracy Gap')
    ax.set_title('Overfitting Indicator')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle('Advanced Training Analysis', fontsize=16)
    plt.tight_layout()
    plt.show()

plot_advanced_training(modern_history)

<a name="systematic-architecture-search"></a>
## Systematic Architecture Search

Let's systematically compare different architectures:

<a name="defining-search-space"></a>
### Defining Search Space

In [ ]:
# Define architecture configurations
architecture_configs = [
    {'name': 'Tiny', 'hidden_sizes': [16], 'dropout': 0.1},
    {'name': 'Shallow', 'hidden_sizes': [32], 'dropout': 0.2},
    {'name': 'Medium', 'hidden_sizes': [64, 32], 'dropout': 0.3},
    {'name': 'Deep', 'hidden_sizes': [128, 64, 32], 'dropout': 0.4},
    {'name': 'Very Deep', 'hidden_sizes': [128, 64, 32, 16], 'dropout': 0.5},
    {'name': 'Wide', 'hidden_sizes': [128, 128], 'dropout': 0.3},
    {'name': 'Pyramid', 'hidden_sizes': [128, 64, 32, 16, 8], 'dropout': 0.4}
]

<a name="running-experiments"></a>
### Running Experiments

In [ ]:
class ArchitectureSearcher:
    """Systematic architecture search for neural networks."""
    
    def __init__(self, train_loader, val_loader, test_loader, device='cpu'):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.results = []
    
    def run_experiment(self, config, epochs=150, lr=0.001, runs=3):
        """Run experiment with multiple seeds for stability."""
        print(f"\nTesting {config['name']} architecture...")
        
        run_metrics = []
        run_times = []
        
        for run in range(runs):
            # Set seed for reproducibility
            torch.manual_seed(42 + run)
            
            # Create model
            model = ModernNeuralNetwork(
                input_features=30,
                hidden_sizes=config['hidden_sizes'],
                dropout_rate=config['dropout']
            )
            
            # Train model
            start_time = time.time()
            model, history = train_neural_network(
                model, self.train_loader, self.val_loader,
                epochs=epochs, lr=lr, device=self.device,
                patience=15, lr_patience=10
            )
            train_time = time.time() - start_time
            
            # Evaluate on test set
            metrics, _, _, _ = evaluate_model(model, self.test_loader, self.device)
            
            run_metrics.append(metrics)
            run_times.append(train_time)
        
        # Aggregate results
        avg_metrics = {}
        std_metrics = {}
        
        for metric in run_metrics[0].keys():
            values = [m[metric] for m in run_metrics]
            avg_metrics[metric] = np.mean(values)
            std_metrics[metric] = np.std(values)
        
        # Store results
        result = {
            'name': config['name'],
            'architecture': ' → '.join(map(str, [30] + config['hidden_sizes'] + [1])),
            'dropout': config['dropout'],
            'parameters': sum(p.numel() for p in model.parameters()),
            'avg_train_time': np.mean(run_times),
            'convergence_epoch': len(history['train_loss']),
            **{f'avg_{k}': v for k, v in avg_metrics.items()},
            **{f'std_{k}': v for k, v in std_metrics.items()}
        }
        
        self.results.append(result)
        return result
    
    def run_all_experiments(self, configs, **kwargs):
        """Run experiments for all configurations."""
        for config in configs:
            self.run_experiment(config, **kwargs)
        return pd.DataFrame(self.results)

# Run architecture search (this will take a few minutes)
print("Starting Architecture Search...")
print("This will test multiple architectures with 3 runs each.")
print("Expected time: 5-10 minutes depending on hardware.\n")

searcher = ArchitectureSearcher(train_loader, val_loader, test_loader, device)

# For demonstration, let's test a subset of architectures
demo_configs = [
    {'name': 'Shallow', 'hidden_sizes': [32], 'dropout': 0.2},
    {'name': 'Medium', 'hidden_sizes': [64, 32], 'dropout': 0.3},
    {'name': 'Deep', 'hidden_sizes': [128, 64, 32], 'dropout': 0.4}
]

results_df = searcher.run_all_experiments(demo_configs, epochs=100, runs=2)

<a name="analyzing-results"></a>
### Analyzing Results

In [ ]:
# Display results
print("\nArchitecture Search Results:")
print("=" * 80)

# Summary table
summary_columns = ['name', 'architecture', 'parameters', 'avg_accuracy', 'std_accuracy', 
                   'avg_f1', 'avg_train_time']
summary_df = results_df[summary_columns].round(4)
print(summary_df.to_string(index=False))

# Visualize results
def plot_architecture_comparison(results_df):
    """Visualize architecture search results."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy vs Parameters
    ax = axes[0, 0]
    ax.scatter(results_df['parameters'], results_df['avg_accuracy'], s=100)
    for idx, row in results_df.iterrows():
        ax.annotate(row['name'], (row['parameters'], row['avg_accuracy']), 
                    xytext=(5, 5), textcoords='offset points')
    ax.set_xlabel('Number of Parameters')
    ax.set_ylabel('Average Accuracy')
    ax.set_title('Model Complexity vs Performance')
    ax.grid(True, alpha=0.3)
    
    # Training Time vs Accuracy
    ax = axes[0, 1]
    ax.scatter(results_df['avg_train_time'], results_df['avg_accuracy'], s=100)
    for idx, row in results_df.iterrows():
        ax.annotate(row['name'], (row['avg_train_time'], row['avg_accuracy']), 
                    xytext=(5, 5), textcoords='offset points')
    ax.set_xlabel('Average Training Time (seconds)')
    ax.set_ylabel('Average Accuracy')
    ax.set_title('Training Efficiency')
    ax.grid(True, alpha=0.3)
    
    # Performance metrics comparison
    ax = axes[1, 0]
    metrics = ['avg_accuracy', 'avg_precision', 'avg_recall', 'avg_f1']
    x = np.arange(len(results_df))
    width = 0.2
    
    for i, metric in enumerate(metrics):
        ax.bar(x + i*width, results_df[metric], width, 
               label=metric.replace('avg_', '').capitalize())
    
    ax.set_xlabel('Architecture')
    ax.set_ylabel('Score')
    ax.set_title('Performance Metrics Comparison')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(results_df['name'])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Accuracy with error bars
    ax = axes[1, 1]
    ax.bar(results_df['name'], results_df['avg_accuracy'], 
           yerr=results_df['std_accuracy'], capsize=5)
    ax.set_xlabel('Architecture')
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy with Standard Deviation')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Architecture Search Results', fontsize=16)
    plt.tight_layout()
    plt.show()

plot_architecture_comparison(results_df)

# Best architecture
best_idx = results_df['avg_accuracy'].idxmax()
best_arch = results_df.loc[best_idx]
print(f"\nBest Architecture: {best_arch['name']}")
print(f"  Accuracy: {best_arch['avg_accuracy']:.4f} ± {best_arch['std_accuracy']:.4f}")
print(f"  Parameters: {best_arch['parameters']:,}")
print(f"  Training Time: {best_arch['avg_train_time']:.1f} seconds")

<a name="interpretability-and-visualization"></a>
## Interpretability and Visualization

Understanding what neural networks learn is crucial for medical applications:

<a name="hidden-layer-representations"></a>
### Hidden Layer Representations

In [ ]:
def extract_hidden_representations(model, data_loader, device='cpu'):
    """Extract hidden layer activations for visualization."""
    model.eval()
    model = model.to(device)
    
    hidden_representations = []
    labels = []
    
    # Hook to capture hidden layer output
    activation = {}
    def get_activation(name):
        def hook(model, input, output):
            activation[name] = output.detach()
        return hook
    
    # Register hook on first hidden layer
    if isinstance(model, ModernNeuralNetwork):
        # Find first ReLU layer (after first hidden layer)
        for i, module in enumerate(model.network):
            if isinstance(module, nn.ReLU):
                model.network[i].register_forward_hook(get_activation('hidden'))
                break
    
    with torch.no_grad():
        for features, batch_labels in data_loader:
            features = features.to(device)
            _ = model(features)
            
            if 'hidden' in activation:
                hidden_representations.append(activation['hidden'].cpu().numpy())
                labels.extend(batch_labels.numpy())
    
    # Concatenate all batches
    hidden_representations = np.vstack(hidden_representations)
    labels = np.array(labels).flatten()
    
    return hidden_representations, labels

# Extract representations from our best model
print("Extracting hidden representations...")
hidden_reps, true_labels = extract_hidden_representations(modern_model, test_loader, device)
print(f"Hidden representation shape: {hidden_reps.shape}")

# Apply t-SNE for 2D visualization
print("\nApplying t-SNE for dimensionality reduction...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
hidden_2d = tsne.fit_transform(hidden_reps)

# Visualize
plt.figure(figsize=(10, 8))
scatter = plt.scatter(hidden_2d[:, 0], hidden_2d[:, 1], 
                     c=true_labels, cmap='coolwarm', 
                     s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
plt.colorbar(scatter, label='True Class')
plt.title('Hidden Layer Representations (t-SNE)', fontsize=14)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

# Add class labels
for class_label in [0, 1]:
    class_points = hidden_2d[true_labels == class_label]
    center = class_points.mean(axis=0)
    plt.annotate(f'Class {class_label}', center, fontsize=12, weight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.grid(True, alpha=0.3)
plt.show()

# Calculate separation quality
from sklearn.metrics import silhouette_score
separation_score = silhouette_score(hidden_2d, true_labels)
print(f"\nSeparation quality (Silhouette Score): {separation_score:.3f}")
print("(Higher is better, range: -1 to 1)")

<a name="feature-importance"></a>
### Feature Importance

In [ ]:
def compute_feature_importance(model, data_loader, device='cpu'):
    """Compute feature importance using gradient-based method."""
    model.eval()
    model = model.to(device)
    
    feature_gradients = []
    
    for features, labels in data_loader:
        features = features.to(device)
        features.requires_grad = True
        
        # Forward pass
        outputs = model(features)
        
        # Backward pass for each sample
        for i in range(features.size(0)):
            model.zero_grad()
            outputs[i].backward(retain_graph=True)
            
            # Get gradients
            grad = features.grad[i].abs().cpu().numpy()
            feature_gradients.append(grad)
    
    # Average gradients across all samples
    feature_importance = np.mean(feature_gradients, axis=0)
    
    return feature_importance

# Compute feature importance
print("Computing feature importance...")
importance_scores = compute_feature_importance(modern_model, test_loader, device)

# Get feature names
cancer_data = load_breast_cancer()
feature_names = cancer_data.feature_names

# Create importance dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance_scores
}).sort_values('importance', ascending=False)

# Visualize top features
plt.figure(figsize=(10, 8))
top_n = 15
top_features = importance_df.head(top_n)

bars = plt.barh(range(top_n), top_features['importance'])
plt.yticks(range(top_n), top_features['feature'])
plt.xlabel('Feature Importance Score')
plt.title(f'Top {top_n} Most Important Features (Neural Network)', fontsize=14)
plt.gca().invert_yaxis()

# Color bars by feature type
for i, (idx, row) in enumerate(top_features.iterrows()):
    if 'worst' in row['feature']:
        bars[i].set_color('red')
    elif 'mean' in row['feature']:
        bars[i].set_color('blue')
    else:
        bars[i].set_color('green')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Worst'),
    Patch(facecolor='blue', label='Mean'),
    Patch(facecolor='green', label='SE')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
for idx, row in importance_df.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

<a name="activation-patterns"></a>
### Activation Patterns

In [ ]:
def analyze_activation_patterns(model, data_loader, device='cpu'):
    """Analyze neuron activation patterns."""
    model.eval()
    model = model.to(device)
    
    # Get a batch of data
    features, labels = next(iter(data_loader))
    features = features.to(device)
    
    # Extract activations at each layer
    activations = []
    
    def hook_fn(module, input, output):
        activations.append(output.detach().cpu().numpy())
    
    # Register hooks
    hooks = []
    for module in model.network:
        if isinstance(module, (nn.Linear, nn.ReLU)):
            hook = module.register_forward_hook(hook_fn)
            hooks.append(hook)
    
    # Forward pass
    with torch.no_grad():
        _ = model(features)
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    # Visualize activation patterns
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    # Select samples to visualize
    sample_indices = [0, 1, 2]  # First three samples
    
    for idx, sample_idx in enumerate(sample_indices):
        ax = axes[idx]
        
        # Plot activation pattern for this sample
        sample_activations = []
        layer_names = []
        
        for i, act in enumerate(activations):
            if len(act.shape) == 2:  # Only process 2D activations
                sample_act = act[sample_idx]
                sample_activations.append(sample_act)
                layer_names.append(f'Layer {i//2}')
        
        # Create heatmap
        if sample_activations:
            activation_matrix = np.array(sample_activations)
            im = ax.imshow(activation_matrix, aspect='auto', cmap='RdBu_r')
            ax.set_yticks(range(len(layer_names)))
            ax.set_yticklabels(layer_names)
            ax.set_xlabel('Neuron Index')
            ax.set_title(f'Sample {sample_idx} (Label: {labels[sample_idx].item():.0f})')
            plt.colorbar(im, ax=ax)
    
    # Activation statistics
    for idx, act in enumerate(activations[::2]):  # Only Linear layers
        if idx < 3:
            ax = axes[3 + idx]
            
            # Plot activation distribution
            ax.hist(act.flatten(), bins=50, alpha=0.7, density=True)
            ax.set_xlabel('Activation Value')
            ax.set_ylabel('Density')
            ax.set_title(f'Layer {idx} Activation Distribution')
            ax.grid(True, alpha=0.3)
            
            # Add statistics
            mean_act = act.mean()
            std_act = act.std()
            ax.axvline(mean_act, color='red', linestyle='--', label=f'Mean: {mean_act:.3f}')
            ax.axvline(mean_act + std_act, color='orange', linestyle=':', label=f'Std: {std_act:.3f}')
            ax.axvline(mean_act - std_act, color='orange', linestyle=':')
            ax.legend()
    
    plt.suptitle('Neural Network Activation Analysis', fontsize=16)
    plt.tight_layout()
    plt.show()

analyze_activation_patterns(modern_model, test_loader, device)

<a name="comprehensive-evaluation"></a>
## Comprehensive Evaluation

<a name="performance-comparison"></a>
### Performance Comparison

Let's compare all our models comprehensively:

In [ ]:
# Evaluate all models
models_to_compare = [
    ('Logistic Regression', logistic_model),
    ('Simple NN (16 hidden)', simple_model),
    ('Modern NN (64→32)', modern_model)
]

comparison_results = []

for name, model in models_to_compare:
    metrics, predictions, probabilities, labels = evaluate_model(model, test_loader, device)
    
    # Add model info
    metrics['model'] = name
    metrics['parameters'] = sum(p.numel() for p in model.parameters())
    
    comparison_results.append(metrics)

# Create comparison dataframe
comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df[['model', 'parameters', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']]

print("\nModel Performance Comparison:")
print("=" * 80)
print(comparison_df.round(4).to_string(index=False))

# Visualize comparison
def plot_model_comparison(comparison_df):
    """Create comprehensive model comparison visualizations."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Performance metrics
    ax = axes[0, 0]
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    x = np.arange(len(comparison_df))
    width = 0.2
    
    for i, metric in enumerate(metrics):
        ax.bar(x + i*width, comparison_df[metric], width, label=metric.capitalize())
    
    ax.set_xlabel('Model')
    ax.set_ylabel('Score')
    ax.set_title('Performance Metrics Comparison')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(comparison_df['model'], rotation=15)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0.9, 1.0)
    
    # ROC AUC comparison
    ax = axes[0, 1]
    ax.bar(comparison_df['model'], comparison_df['roc_auc'], color='purple')
    ax.set_xlabel('Model')
    ax.set_ylabel('ROC AUC Score')
    ax.set_title('ROC AUC Comparison')
    ax.set_xticklabels(comparison_df['model'], rotation=15)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0.95, 1.0)
    
    # Parameters vs Performance
    ax = axes[1, 0]
    scatter = ax.scatter(comparison_df['parameters'], comparison_df['accuracy'], 
                        s=200, c=range(len(comparison_df)), cmap='viridis')
    
    for idx, row in comparison_df.iterrows():
        ax.annotate(row['model'], (row['parameters'], row['accuracy']),
                   xytext=(10, 5), textcoords='offset points', fontsize=10)
    
    ax.set_xlabel('Number of Parameters')
    ax.set_ylabel('Accuracy')
    ax.set_title('Model Complexity vs Performance')
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')
    
    # Performance improvement
    ax = axes[1, 1]
    baseline = comparison_df.iloc[0]['accuracy']  # Logistic regression
    improvements = [(row['accuracy'] - baseline) * 100 for _, row in comparison_df.iterrows()]
    
    bars = ax.bar(comparison_df['model'], improvements)
    bars[0].set_color('gray')  # Baseline
    for i in range(1, len(bars)):
        bars[i].set_color('green' if improvements[i] > 0 else 'red')
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Model')
    ax.set_ylabel('Improvement over Baseline (%)')
    ax.set_title('Performance Improvement vs Logistic Regression')
    ax.set_xticklabels(comparison_df['model'], rotation=15)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Comprehensive Model Comparison', fontsize=16)
    plt.tight_layout()
    plt.show()

plot_model_comparison(comparison_df)

<a name="statistical-significance"></a>
### Statistical Significance

In [ ]:
# McNemar's test for paired model comparison
from statsmodels.stats.contingency_tables import mcnemar

def compare_models_statistically(model1, model2, test_loader, device='cpu'):
    """Perform McNemar's test to compare two models."""
    # Get predictions from both models
    _, pred1, _, labels = evaluate_model(model1, test_loader, device)
    _, pred2, _, _ = evaluate_model(model2, test_loader, device)
    
    # Create contingency table
    # a: both correct, b: model1 correct/model2 wrong
    # c: model1 wrong/model2 correct, d: both wrong
    a = np.sum((pred1 == labels) & (pred2 == labels))
    b = np.sum((pred1 == labels) & (pred2 != labels))
    c = np.sum((pred1 != labels) & (pred2 == labels))
    d = np.sum((pred1 != labels) & (pred2 != labels))
    
    contingency_table = np.array([[a, b], [c, d]])
    
    # Perform test
    result = mcnemar(contingency_table, exact=True)
    
    return contingency_table, result

# Compare logistic regression with neural network
print("Statistical Comparison (McNemar's Test):")
print("=" * 50)

cont_table, test_result = compare_models_statistically(
    logistic_model, modern_model, test_loader, device
)

print("\nContingency Table:")
print("                  Modern NN Correct | Modern NN Wrong")
print(f"LR Correct:               {cont_table[0, 0]:3d}      |      {cont_table[0, 1]:3d}")
print(f"LR Wrong:                 {cont_table[1, 0]:3d}      |      {cont_table[1, 1]:3d}")

print(f"\nMcNemar's test statistic: {test_result.statistic:.3f}")
print(f"p-value: {test_result.pvalue:.4f}")

if test_result.pvalue < 0.05:
    print("\nResult: Statistically significant difference between models (p < 0.05)")
else:
    print("\nResult: No statistically significant difference between models (p >= 0.05)")

<a name="error-analysis"></a>
### Error Analysis

In [ ]:
# Analyze misclassified samples
def analyze_errors(model, test_loader, feature_names, device='cpu'):
    """Analyze misclassified samples."""
    model.eval()
    
    misclassified_features = []
    misclassified_labels = []
    misclassified_predictions = []
    misclassified_probabilities = []
    
    all_features = []
    
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            
            probabilities = model(features)
            predictions = (probabilities > 0.5).float()
            
            # Find misclassified
            misclassified_mask = (predictions != labels.to(device)).squeeze()
            
            if misclassified_mask.any():
                misclassified_features.append(features[misclassified_mask].cpu().numpy())
                misclassified_labels.extend(labels[misclassified_mask].numpy())
                misclassified_predictions.extend(predictions[misclassified_mask].cpu().numpy())
                misclassified_probabilities.extend(probabilities[misclassified_mask].cpu().numpy())
            
            all_features.append(features.cpu().numpy())
    
    if misclassified_features:
        misclassified_features = np.vstack(misclassified_features)
        all_features = np.vstack(all_features)
        
        print(f"\nMisclassified samples: {len(misclassified_labels)}")
        print("\nMisclassification Analysis:")
        
        # Analyze confidence
        misclassified_probs = np.array(misclassified_probabilities).flatten()
        print(f"\nConfidence on errors:")
        print(f"  Mean: {misclassified_probs.mean():.3f}")
        print(f"  Std: {misclassified_probs.std():.3f}")
        print(f"  Min: {misclassified_probs.min():.3f}")
        print(f"  Max: {misclassified_probs.max():.3f}")
        
        # Feature analysis
        print("\nFeature differences (misclassified vs all):")
        feature_diffs = []
        
        for i, feature_name in enumerate(feature_names[:5]):  # Top 5 features
            misclass_mean = misclassified_features[:, i].mean()
            all_mean = all_features[:, i].mean()
            diff = (misclass_mean - all_mean) / all_mean * 100
            feature_diffs.append((feature_name, diff))
            print(f"  {feature_name}: {diff:+.1f}%")
        
        # Visualize error distribution
        plt.figure(figsize=(12, 5))
        
        # Confidence distribution
        plt.subplot(1, 2, 1)
        plt.hist(misclassified_probs, bins=20, alpha=0.7, color='red', edgecolor='black')
        plt.axvline(0.5, color='black', linestyle='--', label='Decision threshold')
        plt.xlabel('Predicted Probability')
        plt.ylabel('Count')
        plt.title('Confidence Distribution of Misclassified Samples')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Error types
        plt.subplot(1, 2, 2)
        false_positives = sum(1 for l, p in zip(misclassified_labels, misclassified_predictions) 
                             if l == 0 and p == 1)
        false_negatives = len(misclassified_labels) - false_positives
        
        plt.bar(['False Positives', 'False Negatives'], [false_positives, false_negatives],
               color=['orange', 'red'])
        plt.ylabel('Count')
        plt.title('Types of Errors')
        plt.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No misclassified samples!")

# Analyze errors for our best model
print("Error Analysis for Modern Neural Network:")
analyze_errors(modern_model, test_loader, feature_names, device)

<a name="production-considerations"></a>
## Production Considerations

<a name="model-complexity-trade-offs"></a>
### Model Complexity Trade-offs

In [ ]:
# Compare model characteristics for production
def analyze_production_metrics(models_dict, test_loader, device='cpu'):
    """Analyze models for production deployment."""
    production_metrics = []
    
    for name, model in models_dict.items():
        # Model size
        param_count = sum(p.numel() for p in model.parameters())
        model_size_mb = param_count * 4 / (1024 * 1024)  # 32-bit floats
        
        # Inference time
        model.eval()
        model = model.to(device)
        
        # Warm-up
        dummy_input = torch.randn(1, 30).to(device)
        for _ in range(10):
            _ = model(dummy_input)
        
        # Time inference
        import time
        n_iterations = 100
        start_time = time.time()
        
        with torch.no_grad():
            for _ in range(n_iterations):
                _ = model(dummy_input)
        
        inference_time_ms = (time.time() - start_time) / n_iterations * 1000
        
        # Performance
        metrics, _, _, _ = evaluate_model(model, test_loader, device)
        
        production_metrics.append({
            'Model': name,
            'Parameters': param_count,
            'Size (MB)': model_size_mb,
            'Inference (ms)': inference_time_ms,
            'Accuracy': metrics['accuracy'],
            'F1 Score': metrics['f1']
        })
    
    return pd.DataFrame(production_metrics)

# Analyze production metrics
models_dict = {
    'Logistic Regression': logistic_model,
    'Simple NN': simple_model,
    'Modern NN': modern_model
}

production_df = analyze_production_metrics(models_dict, test_loader, device)
print("\nProduction Deployment Analysis:")
print("=" * 70)
print(production_df.round(4).to_string(index=False))

# Visualize trade-offs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Accuracy vs Size
scatter = ax1.scatter(production_df['Size (MB)'], production_df['Accuracy'], 
                     s=200, c=range(len(production_df)), cmap='viridis')
for idx, row in production_df.iterrows():
    ax1.annotate(row['Model'], (row['Size (MB)'], row['Accuracy']),
                xytext=(5, 5), textcoords='offset points')
ax1.set_xlabel('Model Size (MB)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs Model Size')
ax1.grid(True, alpha=0.3)

# Accuracy vs Inference Time
scatter = ax2.scatter(production_df['Inference (ms)'], production_df['Accuracy'], 
                     s=200, c=range(len(production_df)), cmap='viridis')
for idx, row in production_df.iterrows():
    ax2.annotate(row['Model'], (row['Inference (ms)'], row['Accuracy']),
                xytext=(5, 5), textcoords='offset points')
ax2.set_xlabel('Inference Time (ms)')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy vs Inference Speed')
ax2.grid(True, alpha=0.3)

plt.suptitle('Production Trade-offs', fontsize=16)
plt.tight_layout()
plt.show()

<a name="when-to-use-neural-networks"></a>
### When to Use Neural Networks

Based on our experiments, here are guidelines for choosing between logistic regression and neural networks:

In [ ]:
# Create decision framework
print("\nDecision Framework: Logistic Regression vs Neural Networks")
print("=" * 60)

decision_factors = {
    'Factor': [
        'Dataset Size',
        'Feature Relationships',
        'Interpretability Need',
        'Inference Speed',
        'Development Time',
        'Model Size',
        'Performance Ceiling'
    ],
    'Logistic Regression': [
        'Works well with small datasets (<1000 samples)',
        'Assumes linear relationships',
        'Highly interpretable coefficients',
        'Very fast (<0.1ms)',
        'Quick to implement and tune',
        'Minimal (31 parameters)',
        'Limited by linearity assumption'
    ],
    'Neural Networks': [
        'Benefits from larger datasets (>10,000 samples)',
        'Can learn non-linear patterns',
        'Black box, requires special techniques',
        'Slower (0.1-1ms per sample)',
        'Requires more experimentation',
        'Larger (100s-1000s parameters)',
        'Can achieve higher accuracy'
    ]
}

decision_df = pd.DataFrame(decision_factors)
print(decision_df.to_string(index=False))

print("\n\nOur Results Summary:")
print("-" * 60)
print(f"Dataset size: {len(train_dataset) + len(val_dataset) + len(test_dataset)} samples")
print(f"Logistic Regression accuracy: {comparison_df[comparison_df['model'] == 'Logistic Regression']['accuracy'].values[0]:.4f}")
print(f"Best Neural Network accuracy: {comparison_df['accuracy'].max():.4f}")
print(f"Improvement: {(comparison_df['accuracy'].max() - comparison_df[comparison_df['model'] == 'Logistic Regression']['accuracy'].values[0]) * 100:.2f}%")

print("\nConclusion for this dataset:")
print("The minimal improvement (~1-2%) suggests that for this well-behaved,")
print("relatively small dataset with good linear separability, logistic")
print("regression is likely the better choice due to its simplicity,")
print("interpretability, and efficiency.")

<a name="conclusion"></a>
## Conclusion

<a name="key-takeaways"></a>
### Key Takeaways

Through our systematic exploration of neural networks for breast cancer detection, we've learned:

1. **Modest Performance Gains**
   - Logistic Regression: ~96.5% accuracy
   - Simple Neural Network: ~97.0% accuracy
   - Modern Neural Network: ~97.5% accuracy
   - Improvement: 1-2% over baseline

2. **Architecture Insights**
   - Deeper isn't always better for small datasets
   - Regularization (dropout, weight decay) helps prevent overfitting
   - Modern techniques (batch norm, learning rate scheduling) improve training

3. **Practical Considerations**
   - 10-20x more parameters than logistic regression
   - 2-5x longer training time
   - More hyperparameters to tune
   - Less interpretable predictions

4. **When Neural Networks Excel**
   - Large datasets (>10,000 samples)
   - Complex, non-linear relationships
   - High-dimensional data (images, text)
   - When 1-2% improvement justifies complexity

5. **PyTorch Best Practices**
   - Modular architecture design
   - Proper train/val/test splits
   - Early stopping with patience
   - Systematic hyperparameter search

<a name="looking-forward"></a>
### Looking Forward

This lesson demonstrated that neural networks aren't always the answer. For our breast cancer dataset:
- Features already well-engineered by domain experts
- Linear relationships capture most patterns
- Small dataset limits deep learning potential

Neural networks truly shine with:
- **Raw data**: Images, audio, text
- **Large scale**: Millions of samples
- **Complex patterns**: Non-linear, hierarchical relationships

In the next lessons, we'll explore:
- **Convolutional Neural Networks**: For image data
- **Recurrent Neural Networks**: For sequential data
- **Transformers**: State-of-the-art architectures

### Next lesson: Advanced architectures for specialized data types

<a name="further-reading"></a>
### Further Reading

To deepen your understanding:

1. **Neural Network Theory**
   - [Deep Learning](https://www.deeplearningbook.org/) by Goodfellow, Bengio, and Courville
   - [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/) by Michael Nielsen

2. **PyTorch Resources**
   - [PyTorch Tutorials](https://pytorch.org/tutorials/)
   - [Dive into Deep Learning](https://d2l.ai/) - PyTorch edition

3. **Medical AI Applications**
   - [AI in Medicine](https://www.nature.com/articles/s41591-018-0300-7)
   - [Deep Learning in Medical Image Analysis](https://www.annualreviews.org/doi/10.1146/annurev-bioeng-071516-044442)

4. **Advanced Topics**
   - [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Transformers
   - [Batch Normalization](https://arxiv.org/abs/1502.03167)
   - [Dropout](https://jmlr.org/papers/v15/srivastava14a.html)

Remember: The best model is the simplest one that solves your problem effectively!

### Thanks for learning!

This notebook is part of the Supervised Machine Learning from First Principles series.

© 2025 Powell-Clark Limited. Licensed under Apache License 2.0.

If you found this helpful, please cite as:
```
Powell-Clark (2025). Supervised Machine Learning from First Principles.
GitHub: https://github.com/powell-clark/supervised-machine-learning
```

Questions or feedback? Contact emmanuel@powellclark.com